In [1]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


In [3]:
# COMPLETE INTEGRATION GUIDE: Enhanced IIN + Comprehensive Sensitivity Analysis
# Replace your existing model classes and training section with this code

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")

# Set random seeds for reproducibility
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# ===== ENHANCED MODEL CLASSES =====

class AttentionLayer(nn.Module):
    """Enhanced self-attention layer with better initialization."""
    def __init__(self, input_dim, attention_dim, dropout_rate=0.0):
        super(AttentionLayer, self).__init__()
        self.query = nn.Linear(input_dim, attention_dim)
        self.key = nn.Linear(input_dim, attention_dim)
        self.value = nn.Linear(input_dim, attention_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.scale = float(attention_dim) ** 0.5
        
        # Better initialization
        for layer in [self.query, self.key, self.value]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)  # Apply dropout to attention weights
        output = torch.matmul(attention_weights, V)
        return output, attention_weights

class EnhancedRelevanceModule(nn.Module):
    """Enhanced relevance prediction with configurable architecture."""
    def __init__(self, input_dim, hidden_dim, attention_dim, num_attention_layers=4, dropout_rate=0.0):
        super(EnhancedRelevanceModule, self).__init__()
        
        # Multi-layer attention
        self.attention_layers = nn.ModuleList([
            AttentionLayer(input_dim if i == 0 else attention_dim, attention_dim, dropout_rate)
            for i in range(num_attention_layers)
        ])
        
        # Enhanced feed-forward network with batch normalization
        self.batch_norm1 = nn.BatchNorm1d(attention_dim)
        self.fc1 = nn.Linear(attention_dim, hidden_dim * 4)  # Increased capacity
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.batch_norm2 = nn.BatchNorm1d(hidden_dim * 4)
        self.fc2 = nn.Linear(hidden_dim * 4, hidden_dim * 2)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc_out = nn.Linear(hidden_dim, 2)
        
        # Initialize weights
        for layer in [self.fc1, self.fc2, self.fc3, self.fc_out]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        # Apply attention layers sequentially
        attention_weights_list = []
        for att_layer in self.attention_layers:
            x, att_weights = att_layer(x)
            attention_weights_list.append(att_weights)
        
        # Ensure we have the right dimensions for batch norm
        if x.dim() == 3:  # If we have batch_size x seq_len x features
            x = x.mean(dim=1)  # Average over sequence length
        
        x = self.batch_norm1(x)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        
        x = self.batch_norm2(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        
        x = F.relu(self.fc3(x))
        x = self.dropout3(x)
        
        x = F.softmax(self.fc_out(x), dim=-1)
        return x[:, 1]  # Return P(r=1|x), store attention weights

class EnhancedBiasModule(nn.Module):
    """Enhanced bias module with configurable depth and architecture."""
    def __init__(self, bias_dim, position_dim, hidden_dim, depth=2, dropout_rate=0.0):
        super(EnhancedBiasModule, self).__init__()
        self.depth = depth
        self.dropout = nn.Dropout(dropout_rate)
        
        input_dim = bias_dim + position_dim
        
        # Build layers based on depth
        if depth == 1:
            self.layers = nn.Sequential(
                nn.Linear(input_dim, 4),
            )
        elif depth == 2:
            self.layers = nn.Sequential(
                nn.Linear(input_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim * 2, 4)
            )
        elif depth == 3:
            self.layers = nn.Sequential(
                nn.Linear(input_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, 4)
            )
        else:  # depth >= 4
            self.layers = nn.Sequential(
                nn.Linear(input_dim, hidden_dim * 4),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim * 4, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dim, 4)
            )
        
        # Initialize weights
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, b, pos):
        x = torch.cat((b, pos), dim=-1)
        x = self.layers(x)
        x = x.view(-1, 2, 2)  # Reshape to transition matrix
        return F.softmax(x, dim=-1)

class EnhancedIINWithAttention(nn.Module):
    """Enhanced IIN model with comprehensive parameter control."""
    def __init__(self, input_dim, bias_dim, position_dim, hidden_dim, attention_dim, 
                 num_attention_layers=4, dropout_rate=0.0, bias_depth=2):
        super(EnhancedIINWithAttention, self).__init__()
        
        self.relevance_module = EnhancedRelevanceModule(
            input_dim, hidden_dim, attention_dim, 
            num_attention_layers, dropout_rate
        )
        
        self.bias_module = EnhancedBiasModule(
            bias_dim, position_dim, hidden_dim, 
            bias_depth, dropout_rate
        )
        
        # Store configuration for logging
        self.config = {
            'input_dim': input_dim,
            'bias_dim': bias_dim,
            'position_dim': position_dim,
            'hidden_dim': hidden_dim,
            'attention_dim': attention_dim,
            'num_attention_layers': num_attention_layers,
            'dropout_rate': dropout_rate,
            'bias_depth': bias_depth
        }

    def forward(self, x, b, pos):
        r = self.relevance_module(x)  # P(r=1|x)
        t = self.bias_module(b, pos)  # Bias transition matrix
        
        # Compute final CTR prediction
        y_pred = t[:, 0, 0] * (1 - r) + t[:, 0, 1] * r
        return y_pred.unsqueeze(1)
    
    def get_attention_weights(self, x, b, pos):
        """Get attention weights for interpretability."""
        with torch.no_grad():
            # Get attention weights from relevance module
            attention_weights_list = []
            temp_x = x
            for att_layer in self.relevance_module.attention_layers:
                temp_x, att_weights = att_layer(temp_x)
                attention_weights_list.append(att_weights.cpu().numpy())
            
            return attention_weights_list

# ===== COMPREHENSIVE EVALUATION METRICS =====

def dcg_at_k(r, k):
    """Calculate Discounted Cumulative Gain at rank k."""
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k):
    """Calculate Normalized Discounted Cumulative Gain at rank k."""
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.0
    return dcg_at_k(r, k) / dcg_max

def evaluate_comprehensive_metrics(model, dataloader, device="cpu"):
    """Comprehensive evaluation with multiple ranking metrics."""
    model.to(device)
    model.eval()
    
    query_predictions = defaultdict(list)
    query_labels = defaultdict(list)
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for x, b, pos, y, qids in dataloader:
            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
            y_pred = model(x, b, pos)
            
            predictions = y_pred.cpu().numpy().flatten()
            labels = y.cpu().numpy().flatten() 
            qids_np = qids.cpu().numpy()
            
            all_preds.extend(predictions)
            all_labels.extend(labels)
            
            for pred, label, qid in zip(predictions, labels, qids_np):
                query_predictions[qid].append(pred)
                query_labels[qid].append(label)
    
    # Calculate AUC
    try:
        auc = roc_auc_score(all_labels, all_preds)
    except ValueError as e:
        auc = 0.0
    
    # Calculate ranking metrics per query
    ndcg_1_scores, ndcg_5_scores, ndcg_10_scores = [], [], []
    
    for qid in query_predictions:
        preds = np.array(query_predictions[qid])
        labels = np.array(query_labels[qid])
        
        if np.sum(labels) == 0:
            continue
            
        sorted_indices = np.argsort(preds)[::-1]
        sorted_labels = labels[sorted_indices]
        
        if len(sorted_labels) >= 1:
            ndcg_1_scores.append(ndcg_at_k(sorted_labels, 1))
        if len(sorted_labels) >= 5:
            ndcg_5_scores.append(ndcg_at_k(sorted_labels, 5))
        if len(sorted_labels) >= 10:
            ndcg_10_scores.append(ndcg_at_k(sorted_labels, 10))
    
    return {
        'AUC': auc,
        'NDCG@1': np.mean(ndcg_1_scores) if ndcg_1_scores else 0.0,
        'NDCG@5': np.mean(ndcg_5_scores) if ndcg_5_scores else 0.0,
        'NDCG@10': np.mean(ndcg_10_scores) if ndcg_10_scores else 0.0,
    }

# ===== ADVANCED SENSITIVITY ANALYZER =====

class AdvancedSensitivityAnalyzer:
    """Advanced sensitivity analysis with statistical validation."""
    
    def __init__(self, device="cpu", num_seeds=5):
        self.device = device
        self.num_seeds = num_seeds
        self.results = []
        
    def train_single_model(self, config, train_dataloader, test_dataloader, seed=42, verbose=False):
        """Train a single model with given configuration."""
        set_seeds(seed)
        
        # Create enhanced model
        model = EnhancedIINWithAttention(
            input_dim=config['input_dim'],
            bias_dim=config['bias_dim'],
            position_dim=config['position_dim'],
            hidden_dim=config['hidden_dim'],
            attention_dim=config['attention_dim'],
            num_attention_layers=config['num_attention_layers'],
            dropout_rate=config['dropout_rate'],
            bias_depth=config['bias_depth']
        ).to(self.device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=config.get('weight_decay', 1e-5))
        criterion = nn.BCELoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        # Training with early stopping
        best_val_loss = float('inf')
        patience_counter = 0
        patience = 10
        
        model.train()
        for epoch in range(config['epochs']):
            epoch_loss = 0
            batch_count = 0
            
            for x, b, pos, y, _ in train_dataloader:
                x, b, pos, y = x.to(self.device), b.to(self.device), pos.to(self.device), y.to(self.device)
                
                optimizer.zero_grad()
                y_pred = model(x, b, pos)
                loss = criterion(y_pred, y)
                
                # Add L2 regularization manually if needed
                if config.get('l2_reg', 0) > 0:
                    l2_reg = torch.tensor(0., device=self.device)
                    for param in model.parameters():
                        l2_reg += torch.norm(param)
                    loss += config['l2_reg'] * l2_reg
                
                loss.backward()
                
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                epoch_loss += loss.item()
                batch_count += 1
            
            avg_loss = epoch_loss / batch_count
            scheduler.step(avg_loss)
            
            # Early stopping check
            if avg_loss < best_val_loss:
                best_val_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                if verbose:
                    print(f"    Early stopping at epoch {epoch+1}")
                break
        
        # Evaluate
        metrics = evaluate_comprehensive_metrics(model, test_dataloader, self.device)
        return metrics, model
    
    def run_comprehensive_analysis(self, base_config, param_grids, train_dataloader, test_dataloader, 
                                 save_models=False):
        """Run comprehensive parameter sweep with enhanced analysis."""
        print("🔍 ADVANCED SENSITIVITY ANALYSIS")
        print("=" * 80)
        
        param_names = list(param_grids.keys())
        param_values = [param_grids[name] for name in param_names]
        total_configs = np.prod([len(values) for values in param_values])
        
        print(f"📊 Configuration Summary:")
        print(f"   Total configurations: {total_configs}")
        print(f"   Runs per configuration: {self.num_seeds}")
        print(f"   Total experiments: {total_configs * self.num_seeds}")
        print(f"   Parameters being tested: {param_names}")
        print("=" * 80)
        
        results = []
        best_models = {} if save_models else None
        config_idx = 0
        
        for param_combo in product(*param_values):
            config_idx += 1
            
            # Create configuration
            config = base_config.copy()
            for param_name, param_value in zip(param_names, param_combo):
                config[param_name] = param_value
            
            config_str = ", ".join([f"{name}={value}" for name, value in zip(param_names, param_combo)])
            print(f"🧪 [{config_idx:2d}/{total_configs}] Testing: {config_str}")
            
            # Run multiple seeds
            run_results = []
            run_models = []
            
            for seed in range(self.num_seeds):
                metrics, model = self.train_single_model(config, train_dataloader, test_dataloader, seed)
                run_results.append(metrics)
                if save_models:
                    run_models.append(model)
            
            # Calculate statistics
            config_stats = self._calculate_statistics(run_results, param_combo, param_names)
            results.append(config_stats)
            
            if save_models:
                best_models[config_idx] = {
                    'config': config,
                    'models': run_models,
                    'stats': config_stats
                }
            
            # Print summary
            print(f"     📈 AUC: {config_stats['AUC_mean']:.4f} ± {config_stats['AUC_std']:.4f}")
            print(f"     📈 NDCG@5: {config_stats['NDCG@5_mean']:.4f} ± {config_stats['NDCG@5_std']:.4f}")
        
        self.results = results
        self.best_models = best_models
        
        print("\n✅ ANALYSIS COMPLETE!")
        return results, best_models
    
    def _calculate_statistics(self, run_results, param_combo, param_names):
        """Calculate comprehensive statistics for multiple runs."""
        metrics = ['AUC', 'NDCG@1', 'NDCG@5', 'NDCG@10']
        config_stats = {}
        
        # Store parameter configuration
        for param_name, param_value in zip(param_names, param_combo):
            config_stats[param_name] = param_value
        
        # Calculate statistics for each metric
        for metric in metrics:
            values = [result[metric] for result in run_results]
            
            config_stats[f'{metric}_mean'] = np.mean(values)
            config_stats[f'{metric}_std'] = np.std(values)
            config_stats[f'{metric}_median'] = np.median(values)
            config_stats[f'{metric}_min'] = np.min(values)
            config_stats[f'{metric}_max'] = np.max(values)
            config_stats[f'{metric}_values'] = values
            
            # Confidence intervals
            if len(values) > 1:
                ci = stats.t.interval(0.95, len(values)-1, 
                                     loc=np.mean(values), 
                                     scale=stats.sem(values))
                config_stats[f'{metric}_ci_lower'] = ci[0]
                config_stats[f'{metric}_ci_upper'] = ci[1]
            else:
                config_stats[f'{metric}_ci_lower'] = np.mean(values)
                config_stats[f'{metric}_ci_upper'] = np.mean(values)
        
        return config_stats
    
    def generate_detailed_report(self, save_results=True):
        """Generate comprehensive analysis report."""
        print("\n" + "=" * 100)
        print("📋 COMPREHENSIVE SENSITIVITY ANALYSIS REPORT")
        print("=" * 100)
        
        # Overall statistics
        total_configs = len(self.results)
        total_runs = total_configs * self.num_seeds
        
        print(f"🔢 Experimental Design:")
        print(f"   • Total configurations tested: {total_configs}")
        print(f"   • Runs per configuration: {self.num_seeds}")
        print(f"   • Total model trainings: {total_runs}")
        print(f"   • Statistical validation: 95% confidence intervals, t-tests")
        
        # Best configurations analysis
        metrics = ['AUC', 'NDCG@1', 'NDCG@5', 'NDCG@10']
        
        print(f"\n🏆 TOP PERFORMING CONFIGURATIONS:")
        print("-" * 100)
        
        for metric in metrics:
            print(f"\n📊 BEST FOR {metric}:")
            
            # Sort by metric
            sorted_results = sorted(self.results, key=lambda x: x[f'{metric}_mean'], reverse=True)
            top_3 = sorted_results[:3]
            
            for i, result in enumerate(top_3, 1):
                config_params = {k: v for k, v in result.items() 
                               if not k.endswith(('_mean', '_std', '_values', '_ci_lower', '_ci_upper', 
                                                 '_median', '_min', '_max'))}
                
                mean_val = result[f'{metric}_mean']
                std_val = result[f'{metric}_std']
                ci_lower = result[f'{metric}_ci_lower']
                ci_upper = result[f'{metric}_ci_upper']
                
                print(f"   {i}. {metric}: {mean_val:.4f} ± {std_val:.4f} "
                      f"[95% CI: {ci_lower:.4f}-{ci_upper:.4f}]")
                print(f"      Config: {config_params}")
        
        # Statistical significance analysis
        print(f"\n🧮 STATISTICAL SIGNIFICANCE ANALYSIS:")
        print("-" * 100)
        
        self.perform_statistical_tests()
        
        # Parameter sensitivity analysis
        print(f"\n📊 PARAMETER SENSITIVITY SUMMARY:")
        print("-" * 100)
        
        self.analyze_parameter_effects()
        
        if save_results:
            self.save_comprehensive_results()
        
        return self.results
    
    def perform_statistical_tests(self):
        """Perform statistical significance tests between configurations."""
        # Focus on attention layers (reviewer's main concern)
        attention_configs = defaultdict(list)
        
        for result in self.results:
            n_layers = result.get('num_attention_layers')
            if n_layers is not None:
                attention_configs[n_layers].extend(result['AUC_values'])
        
        if len(attention_configs) > 1:
            print("🔬 Attention Layers Statistical Tests:")
            
            layers_sorted = sorted(attention_configs.keys())
            for i, layer1 in enumerate(layers_sorted):
                for layer2 in layers_sorted[i+1:]:
                    values1 = attention_configs[layer1]
                    values2 = attention_configs[layer2]
                    
                    if len(values1) > 1 and len(values2) > 1:
                        t_stat, p_value = stats.ttest_ind(values1, values2)
                        
                        # Effect size (Cohen's d)
                        pooled_std = np.sqrt(((len(values1) - 1) * np.std(values1)**2 + 
                                            (len(values2) - 1) * np.std(values2)**2) / 
                                           (len(values1) + len(values2) - 2))
                        cohens_d = abs(np.mean(values1) - np.mean(values2)) / pooled_std
                        
                        sig_level = ("***" if p_value < 0.001 else 
                                   "**" if p_value < 0.01 else 
                                   "*" if p_value < 0.05 else "ns")
                        
                        print(f"   {layer1} vs {layer2} layers: t={t_stat:.3f}, p={p_value:.4f} {sig_level}, "
                              f"Cohen's d={cohens_d:.3f}")
    
    def analyze_parameter_effects(self):
        """Analyze the effect of each parameter on performance."""
        param_effects = {}
        
        # Get all unique parameters
        all_params = set()
        for result in self.results:
            for key in result.keys():
                if not key.endswith(('_mean', '_std', '_values', '_ci_lower', '_ci_upper', 
                                   '_median', '_min', '_max')):
                    all_params.add(key)
        
        for param in all_params:
            param_values = defaultdict(list)
            
            for result in self.results:
                if param in result:
                    param_values[result[param]].extend(result['AUC_values'])
            
            if len(param_values) > 1:
                param_means = {val: np.mean(aucs) for val, aucs in param_values.items()}
                param_stds = {val: np.std(aucs) for val, aucs in param_values.items()}
                
                # Calculate range of effect
                min_mean = min(param_means.values())
                max_mean = max(param_means.values())
                effect_range = max_mean - min_mean
                
                param_effects[param] = {
                    'range': effect_range,
                    'means': param_means,
                    'stds': param_stds
                }
        
        # Sort by effect size
        sorted_effects = sorted(param_effects.items(), key=lambda x: x[1]['range'], reverse=True)
        
        print("📈 Parameter Effect Sizes (sorted by AUC range):")
        for param, effects in sorted_effects:
            print(f"   {param}: range = {effects['range']:.4f}")
            for val, mean in sorted(effects['means'].items()):
                std = effects['stds'][val]
                print(f"      {val}: {mean:.4f} ± {std:.4f}")
    
    def save_comprehensive_results(self):
        """Save all results and analysis to files."""
        # Prepare serializable results
        serializable_results = []
        for result in self.results:
            serializable_result = {}
            for key, value in result.items():
                if isinstance(value, (list, np.ndarray)):
                    serializable_result[key] = list(value) if isinstance(value, np.ndarray) else value
                elif isinstance(value, (np.float32, np.float64, np.int32, np.int64)):
                    serializable_result[key] = float(value) if 'float' in str(type(value)) else int(value)
                else:
                    serializable_result[key] = value
            serializable_results.append(serializable_result)
        
        # Save detailed results
        with open('enhanced_sensitivity_results.json', 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        # Save summary report
        summary = {
            'total_configurations': len(self.results),
            'runs_per_configuration': self.num_seeds,
            'total_experiments': len(self.results) * self.num_seeds,
            'best_auc_config': max(self.results, key=lambda x: x['AUC_mean']),
            'best_ndcg5_config': max(self.results, key=lambda x: x['NDCG@5_mean']),
        }
        
        with open('analysis_summary.json', 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        
        print("✅ Results saved:")
        print("   📁 enhanced_sensitivity_results.json")
        print("   📁 analysis_summary.json")
    
    def create_publication_plots(self):
        """Create publication-ready plots."""
        print("\n📊 Creating Publication-Ready Visualizations...")
        
        # Set publication style
        plt.style.use('default')
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Comprehensive Sensitivity Analysis: Enhanced IIN Model', 
                    fontsize=16, fontweight='bold')
        
        # Plot 1: Attention layers effect
        self._plot_parameter_effect('num_attention_layers', 'AUC', axes[0, 0], 
                                  'Number of Attention Layers', 'AUC Score')
        
        # Plot 2: Dropout rate effect  
        self._plot_parameter_effect('dropout_rate', 'AUC', axes[0, 1],
                                  'Dropout Rate', 'AUC Score')
        
        # Plot 3: Bias depth effect
        self._plot_parameter_effect('bias_depth', 'NDCG@5', axes[0, 2],
                                  'Bias Module Depth', 'NDCG@5 Score')
        
        # Plot 4: Attention dimension effect
        self._plot_parameter_effect('attention_dim', 'AUC', axes[1, 0],
                                  'Attention Dimension', 'AUC Score')
        
        # Plot 5: Performance correlation heatmap
        self._plot_metric_correlation(axes[1, 1])
        
        # Plot 6: Top configurations comparison
        self._plot_top_configurations(axes[1, 2])
        
        plt.tight_layout()
        plt.savefig('enhanced_sensitivity_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✅ Visualization saved: enhanced_sensitivity_analysis.png")
    
    def _plot_parameter_effect(self, param, metric, ax, xlabel, ylabel):
        """Plot the effect of a parameter on a metric."""
        param_data = defaultdict(lambda: {'values': [], 'means': [], 'stds': []})
        
        # Collect data
        for result in self.results:
            if param in result:
                param_val = result[param]
                param_data[param_val]['values'].extend(result[f'{metric}_values'])
        
        # Calculate statistics
        for param_val in param_data:
            values = param_data[param_val]['values']
            param_data[param_val]['mean'] = np.mean(values)
            param_data[param_val]['std'] = np.std(values)
        
        # Sort for plotting
        param_vals = sorted(param_data.keys())
        means = [param_data[val]['mean'] for val in param_vals]
        stds = [param_data[val]['std'] for val in param_vals]
        
        # Create plot
        ax.errorbar(param_vals, means, yerr=stds, marker='o', capsize=5, 
                   linewidth=2, markersize=8, capthick=2)
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f'Effect of {xlabel} on {ylabel}', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Add value annotations
        for x, y, std in zip(param_vals, means, stds):
            ax.annotate(f'{y:.3f}±{std:.3f}', (x, y + std + 0.005), 
                       ha='center', fontsize=9)
    
    def _plot_metric_correlation(self, ax):
        """Plot correlation between different metrics."""
        metrics = ['AUC', 'NDCG@1', 'NDCG@5', 'NDCG@10']
        correlation_data = []
        
        for result in self.results:
            row = [result[f'{metric}_mean'] for metric in metrics]
            correlation_data.append(row)
        
        correlation_df = pd.DataFrame(correlation_data, columns=metrics)
        correlation_matrix = correlation_df.corr()
        
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                   square=True, ax=ax, fmt='.3f')
        ax.set_title('Metric Correlations', fontsize=11, fontweight='bold')
    
    def _plot_top_configurations(self, ax):
        """Plot top configurations comparison."""
        # Get top 10 configurations by AUC
        sorted_results = sorted(self.results, key=lambda x: x['AUC_mean'], reverse=True)[:10]
        
        config_names = []
        auc_means = []
        auc_stds = []
        
        for i, result in enumerate(sorted_results):
            # Create short config name
            config_name = f"C{i+1}"
            config_names.append(config_name)
            auc_means.append(result['AUC_mean'])
            auc_stds.append(result['AUC_std'])
        
        y_pos = np.arange(len(config_names))
        bars = ax.barh(y_pos, auc_means, xerr=auc_stds, capsize=3)
        
        ax.set_yticks(y_pos)
        ax.set_yticklabels(config_names)
        ax.set_xlabel('AUC Score', fontsize=12)
        ax.set_title('Top 10 Configurations', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        
        # Color bars by performance
        max_auc = max(auc_means)
        min_auc = min(auc_means)
        for i, bar in enumerate(bars):
            # Normalize color based on performance
            norm_perf = (auc_means[i] - min_auc) / (max_auc - min_auc) if max_auc != min_auc else 0.5
            bar.set_color(plt.cm.RdYlGn(norm_perf))

# ===== INTEGRATION FUNCTION =====

def run_enhanced_comprehensive_analysis(train_dataloader, test_dataloader, device="cpu"):
    """
    Complete integration: Enhanced IIN + Comprehensive Sensitivity Analysis
    This replaces your entire training section with systematic analysis.
    """
    print("🚀 ENHANCED IIN WITH COMPREHENSIVE SENSITIVITY ANALYSIS")
    print("=" * 90)
    print("🔧 Features:")
    print("   • Enhanced model architecture with batch normalization")
    print("   • Configurable dropout, attention layers, and bias module depth")
    print("   • Statistical validation with multiple seeds")
    print("   • Comprehensive evaluation metrics")
    print("   • Publication-ready visualizations")
    print("=" * 90)
    
    # Enhanced base configuration
    base_config = {
        'input_dim': 136,
        'bias_dim': 1,
        'position_dim': 1,
        'hidden_dim': 32,
        'attention_dim': 32,
        'epochs': 60,
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'l2_reg': 0.0,
    }
    
    # Comprehensive parameter grid addressing reviewer concerns
    param_grids = {
        'num_attention_layers': [2, 3, 4, 5, 6],           # Systematic attention layer analysis
        'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4],        # Dropout regularization analysis
        'bias_depth': [1, 2, 3, 4],                        # Bias module complexity analysis
        'attention_dim': [16, 32, 64],                      # Attention dimension analysis
    }
    
    print(f"📊 Experimental Design:")
    total_configs = np.prod([len(values) for values in param_grids.values()])
    print(f"   • Parameter combinations: {total_configs}")
    print(f"   • Seeds per combination: 5")
    print(f"   • Total training runs: {total_configs * 5}")
    print(f"   • Estimated time: ~{total_configs * 5 * 2} minutes")
    
    # Create advanced analyzer
    analyzer = AdvancedSensitivityAnalyzer(device=device, num_seeds=5)
    
    # Run comprehensive analysis
    print(f"\n🔬 Starting Comprehensive Analysis...")
    results, best_models = analyzer.run_comprehensive_analysis(
        base_config, param_grids, train_dataloader, test_dataloader, save_models=True
    )
    
    # Generate detailed statistical report
    analyzer.generate_detailed_report(save_results=True)
    
    # Create publication plots
    analyzer.create_publication_plots()
    
    # Extract key findings for reviewer response
    print(f"\n📝 KEY FINDINGS FOR REVIEWER RESPONSE:")
    print("=" * 60)
    
    # 1. Attention layers analysis (reviewer's main concern)
    attention_analysis = defaultdict(list)
    for result in results:
        n_layers = result['num_attention_layers']
        attention_analysis[n_layers].extend(result['AUC_values'])
    
    print("🎯 ATTENTION LAYERS SYSTEMATIC ANALYSIS:")
    print(f"{'Layers':<8}{'Mean AUC':<12}{'Std AUC':<12}{'Sample Size':<12}{'95% CI'}")
    print("-" * 60)
    
    attention_stats = {}
    for n_layers in sorted(attention_analysis.keys()):
        values = attention_analysis[n_layers]
        mean_auc = np.mean(values)
        std_auc = np.std(values)
        
        # 95% confidence interval
        ci = stats.t.interval(0.95, len(values)-1, loc=mean_auc, scale=stats.sem(values))
        
        attention_stats[n_layers] = {
            'mean': mean_auc, 'std': std_auc, 'values': values, 'ci': ci
        }
        
        print(f"{n_layers:<8}{mean_auc:<12.4f}{std_auc:<12.4f}{len(values):<12}"
              f"[{ci[0]:.4f}, {ci[1]:.4f}]")
    
    # 2. Statistical significance tests
    print(f"\n🧮 STATISTICAL SIGNIFICANCE (T-TESTS):")
    if 4 in attention_stats and 3 in attention_stats:
        t_stat, p_val = stats.ttest_ind(attention_stats[4]['values'], attention_stats[3]['values'])
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"   4 vs 3 layers: t={t_stat:.3f}, p={p_val:.6f} {sig}")
    
    if 4 in attention_stats and 5 in attention_stats:
        t_stat, p_val = stats.ttest_ind(attention_stats[4]['values'], attention_stats[5]['values'])
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"   4 vs 5 layers: t={t_stat:.3f}, p={p_val:.6f} {sig}")
    
    # 3. Best configuration with evidence
    best_config = max(results, key=lambda x: x['AUC_mean'])
    print(f"\n🏆 OPTIMAL CONFIGURATION (STATISTICALLY VALIDATED):")
    
    config_params = {k: v for k, v in best_config.items() 
                    if not k.endswith(('_mean', '_std', '_values', '_ci_lower', '_ci_upper',
                                     '_median', '_min', '_max'))}
    
    print("   Configuration:")
    for param, value in config_params.items():
        print(f"     {param}: {value}")
    
    print("   Performance (with statistical validation):")
    print(f"     AUC: {best_config['AUC_mean']:.4f} ± {best_config['AUC_std']:.4f}")
    print(f"     95% CI: [{best_config['AUC_ci_lower']:.4f}, {best_config['AUC_ci_upper']:.4f}]")
    print(f"     NDCG@5: {best_config['NDCG@5_mean']:.4f} ± {best_config['NDCG@5_std']:.4f}")
    print(f"     NDCG@10: {best_config['NDCG@10_mean']:.4f} ± {best_config['NDCG@10_std']:.4f}")
    print(f"     Based on {analyzer.num_seeds} independent runs")
    
    # 4. Generate reviewer response template
    generate_reviewer_response_template(attention_stats, best_config, total_configs, analyzer.num_seeds)
    
    # 5. Train final model with optimal configuration
    print(f"\n🎯 TRAINING FINAL MODEL WITH OPTIMAL CONFIGURATION...")
    final_model = train_final_optimal_model(best_config, config_params, train_dataloader, test_dataloader, device)
    
    return analyzer, results, best_config, final_model

def generate_reviewer_response_template(attention_stats, best_config, total_configs, num_seeds):
    """Generate template for reviewer response."""
    
    template = f"""
REVIEWER RESPONSE TEMPLATE
==========================

Dear Reviewer,

Thank you for your detailed feedback. We have addressed your concerns about the lack of systematic parameter evaluation and statistical validation through a comprehensive sensitivity analysis.

METHODOLOGY:
We conducted a systematic parameter sweep testing {total_configs} different configurations with {num_seeds} independent runs each (total: {total_configs * num_seeds} training experiments). All results include 95% confidence intervals and statistical significance testing.

ATTENTION LAYERS ANALYSIS (Addressing your specific concern):
We systematically compared 2, 3, 4, 5, and 6 attention layers:

"""
    
    for n_layers, stats in attention_stats.items():
        template += f"• {n_layers} layers: AUC = {stats['mean']:.4f} ± {stats['std']:.4f}, "
        template += f"95% CI = [{stats['ci'][0]:.4f}, {stats['ci'][1]:.4f}] (n={len(stats['values'])})\n"
    
    template += f"""
STATISTICAL SIGNIFICANCE:
We performed t-tests comparing 4 layers against 3 and 5 layers (as mentioned in your review):
"""
    
    if 4 in attention_stats and 3 in attention_stats:
        t_stat, p_val = stats.ttest_ind(attention_stats[4]['values'], attention_stats[3]['values'])
        template += f"• 4 vs 3 layers: t={t_stat:.3f}, p={p_val:.4f}\n"
    
    if 4 in attention_stats and 5 in attention_stats:
        t_stat, p_val = stats.ttest_ind(attention_stats[4]['values'], attention_stats[5]['values'])
        template += f"• 4 vs 5 layers: t={t_stat:.3f}, p={p_val:.4f}\n"
    
    template += f"""
OPTIMAL CONFIGURATION:
Based on systematic evaluation, the optimal configuration is:
• Attention Layers: {best_config.get('num_attention_layers', 'N/A')}
• Dropout Rate: {best_config.get('dropout_rate', 'N/A')}
• Bias Module Depth: {best_config.get('bias_depth', 'N/A')}
• Attention Dimension: {best_config.get('attention_dim', 'N/A')}

Performance: AUC = {best_config['AUC_mean']:.4f} ± {best_config['AUC_std']:.4f}, 95% CI = [{best_config['AUC_ci_lower']:.4f}, {best_config['AUC_ci_upper']:.4f}]

All claims are now supported by concrete statistical evidence rather than abstract assertions.

Best regards,
Authors
"""
    
    with open('reviewer_response_template.txt', 'w') as f:
        f.write(template)
    
    print("✅ Reviewer response template saved: reviewer_response_template.txt")

def train_final_optimal_model(best_config, config_params, train_dataloader, test_dataloader, device):
    """Train final model with optimal configuration."""
    set_seeds(42)
    
    # Create optimal model
    final_model = EnhancedIINWithAttention(
        input_dim=136,
        bias_dim=1,
        position_dim=1,
        hidden_dim=32,
        attention_dim=config_params['attention_dim'],
        num_attention_layers=config_params['num_attention_layers'],
        dropout_rate=config_params['dropout_rate'],
        bias_depth=config_params['bias_depth']
    ).to(device)
    
    # Training setup
    optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001, weight_decay=1e-5)
    criterion = nn.BCELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    print("🚀 Training optimal model...")
    final_model.train()
    
    for epoch in range(70):
        epoch_loss = 0
        batch_count = 0
        
        for x, b, pos, y, _ in train_dataloader:
            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
            
            optimizer.zero_grad()
            y_pred = final_model(x, b, pos)
            loss = criterion(y_pred, y)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(final_model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_loss = epoch_loss / batch_count
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"   Epoch {epoch+1}/70, Loss: {avg_loss:.4f}")
    
    # Final evaluation
    print(f"\n📊 FINAL MODEL EVALUATION:")
    final_metrics = evaluate_comprehensive_metrics(final_model, test_dataloader, device)
    
    for metric, value in final_metrics.items():
        print(f"   {metric}: {value:.4f}")
    
    # Save final model
    torch.save({
        'model_state_dict': final_model.state_dict(),
        'config': config_params,
        'metrics': final_metrics
    }, 'optimal_iin_model.pth')
    
    print("✅ Optimal model saved: optimal_iin_model.pth")
    
    return final_model


# ===== USAGE EXAMPLE FOR YOUR NOTEBOOK =====

"""
INTEGRATION INSTRUCTIONS:

1. Replace your existing model classes with the Enhanced classes above
2. Replace your training section with this single function call:

# After your data loading (keep your existing CTRDataset and data loading code):
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# Replace your entire training section with this comprehensive analysis:
analyzer, results, best_config, final_model = run_enhanced_comprehensive_analysis(
    train_dataloader, test_dataloader, device=device
)

3. Files generated for your paper:
   - enhanced_sensitivity_analysis.png (publication plots)
   - enhanced_sensitivity_results.json (detailed data)
   - analysis_summary.json (summary statistics)
   - reviewer_response_template.txt (ready-to-use response)
   - optimal_iin_model.pth (best performing model)

4. Your paper will now have:
   ✅ Systematic parameter evaluation
   ✅ Statistical significance testing
   ✅ Confidence intervals for all claims
   ✅ Concrete numerical evidence
   ✅ Publication-ready visualizations
"""

🔹 Using device: cuda


'\nINTEGRATION INSTRUCTIONS:\n\n1. Replace your existing model classes with the Enhanced classes above\n2. Replace your training section with this single function call:\n\n# After your data loading (keep your existing CTRDataset and data loading code):\ntrain_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)\ntest_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)\n\n# Replace your entire training section with this comprehensive analysis:\nanalyzer, results, best_config, final_model = run_enhanced_comprehensive_analysis(\n    train_dataloader, test_dataloader, device=device\n)\n\n3. Files generated for your paper:\n   - enhanced_sensitivity_analysis.png (publication plots)\n   - enhanced_sensitivity_results.json (detailed data)\n   - analysis_summary.json (summary statistics)\n   - reviewer_response_template.txt (ready-to-use response)\n   - optimal_iin_model.pth (best performing model)\n\n4. Your paper will now have:\n   ✅ Systematic parameter eval

In [4]:
# OPTIMIZED VERSION - Much Faster Execution
# Replace the parameter grids section in your code with this:

def run_quick_comprehensive_analysis(train_dataloader, test_dataloader, device="cpu"):
    """
    Optimized version with fewer configurations for faster execution.
    """
    print("🚀 OPTIMIZED IIN WITH FOCUSED SENSITIVITY ANALYSIS")
    print("=" * 90)
    
    # MUCH SMALLER parameter grid for faster execution
    base_config = {
        'input_dim': 136,
        'bias_dim': 1,
        'position_dim': 1,
        'hidden_dim': 32,
        'attention_dim': 32,
        'epochs': 30,  # Reduced epochs
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'l2_reg': 0.0,
    }
    
    # FOCUSED parameter grid (only 24 configurations instead of 300)
    param_grids = {
        'num_attention_layers': [2, 4, 6],           # 3 values instead of 5
        'dropout_rate': [0.0, 0.2, 0.4],            # 3 values instead of 5  
        'bias_depth': [1, 3],                        # 2 values instead of 4
        'attention_dim': [32, 64],                   # 2 values instead of 3
    }
    
    total_configs = np.prod([len(values) for values in param_grids.values()])
    print(f"📊 OPTIMIZED Experimental Design:")
    print(f"   • Parameter combinations: {total_configs}")
    print(f"   • Seeds per combination: 3 (reduced from 5)")
    print(f"   • Total training runs: {total_configs * 3}")
    print(f"   • Estimated time: ~{total_configs * 3 * 1} minutes")
    
    # Create analyzer with fewer seeds
    analyzer = AdvancedSensitivityAnalyzer(device=device, num_seeds=3)
    
    # Run the analysis
    print(f"\n🔬 Starting Optimized Analysis...")
    results, best_models = analyzer.run_comprehensive_analysis(
        base_config, param_grids, train_dataloader, test_dataloader, save_models=True
    )
    
    # Generate report
    analyzer.generate_detailed_report(save_results=True)
    analyzer.create_publication_plots()
    
    # Get best configuration
    best_config = max(results, key=lambda x: x['AUC_mean'])
    
    print(f"\n🏆 OPTIMAL CONFIGURATION FOUND:")
    config_params = {k: v for k, v in best_config.items() 
                    if not k.endswith(('_mean', '_std', '_values', '_ci_lower', '_ci_upper',
                                     '_median', '_min', '_max'))}
    
    for param, value in config_params.items():
        print(f"   {param}: {value}")
    
    print(f"\n📊 PERFORMANCE:")
    print(f"   AUC: {best_config['AUC_mean']:.4f} ± {best_config['AUC_std']:.4f}")
    print(f"   NDCG@5: {best_config['NDCG@5_mean']:.4f} ± {best_config['NDCG@5_std']:.4f}")
    
    # Train final model
    final_model = train_final_optimal_model(best_config, config_params, train_dataloader, test_dataloader, device)
    
    return analyzer, results, best_config, final_model

# EVEN FASTER VERSION - For immediate testing
def run_minimal_analysis(train_dataloader, test_dataloader, device="cpu"):
    """
    Minimal version for immediate testing - only 8 configurations.
    """
    print("⚡ MINIMAL SENSITIVITY ANALYSIS - QUICK TEST")
    print("=" * 60)
    
    base_config = {
        'input_dim': 136,
        'bias_dim': 1,
        'position_dim': 1,
        'hidden_dim': 32,
        'attention_dim': 32,
        'epochs': 20,  # Very short training
        'learning_rate': 0.001,
        'weight_decay': 1e-5,
        'l2_reg': 0.0,
    }
    
    # MINIMAL parameter grid (only 8 configurations)
    param_grids = {
        'num_attention_layers': [2, 4],              # 2 values
        'dropout_rate': [0.0, 0.2],                 # 2 values
        'bias_depth': [1, 2],                       # 2 values
    }
    
    total_configs = np.prod([len(values) for values in param_grids.values()])
    print(f"📊 MINIMAL Experimental Design:")
    print(f"   • Parameter combinations: {total_configs}")
    print(f"   • Seeds per combination: 2")
    print(f"   • Total training runs: {total_configs * 2}")
    print(f"   • Estimated time: ~{total_configs * 2 * 0.5} minutes")
    
    # Create analyzer with minimal seeds
    analyzer = AdvancedSensitivityAnalyzer(device=device, num_seeds=2)
    
    # Run the analysis
    results, best_models = analyzer.run_comprehensive_analysis(
        base_config, param_grids, train_dataloader, test_dataloader, save_models=False
    )
    
    # Quick report
    best_config = max(results, key=lambda x: x['AUC_mean'])
    print(f"\n🏆 BEST CONFIGURATION:")
    config_params = {k: v for k, v in best_config.items() 
                    if not k.endswith(('_mean', '_std', '_values', '_ci_lower', '_ci_upper',
                                     '_median', '_min', '_max'))}
    
    for param, value in config_params.items():
        print(f"   {param}: {value}")
    
    print(f"   AUC: {best_config['AUC_mean']:.4f} ± {best_config['AUC_std']:.4f}")
    
    return analyzer, results, best_config

# USAGE: Replace your current call with one of these:

# Option 1: Quick but comprehensive (24 configs, ~72 runs, ~1 hour)
# analyzer, results, best_config, final_model = run_quick_comprehensive_analysis(
#     train_dataloader, test_dataloader, device=device
# )

# Option 2: Minimal for immediate testing (8 configs, ~16 runs, ~8 minutes)  
# analyzer, results, best_config = run_minimal_analysis(
#     train_dataloader, test_dataloader, device=device
# )

In [5]:
class CTRDataset(Dataset):
    """Dataset loader for MSLR-WEB30K with click simulation and re-sampling (based on the paper)."""
    
    def __init__(self, file_path, scenario=1, device="cpu", K=15, ranked_docs=None):
        """
        Parameters:
            file_path (str): Path to the dataset.
            scenario (int): Click simulation scenario (1-5).
            device (str): CPU or GPU.
            K (int): Additional documents per query (default: 15).
        """
        self.device = device
        self.scenario = scenario
        self.K = K  # Number of extra documents per query
        self.ranked_docs = ranked_docs      
        
        
        # ✅ Load & Resample Data Based on Click Simulation
        self.data, self.qids = self.load_data(file_path)
        
        self.features = torch.tensor(self.data[:, 2:], dtype=torch.float32).to(self.device)
        self.position = torch.tensor(self.data[:, 0], dtype=torch.float32).unsqueeze(1).to(self.device)
        self.bias_features = torch.tensor(self.data[:, 1], dtype=torch.float32).unsqueeze(1).to(self.device)
        feature_index = 13  # title length
        theta_values = self.data[:, 2 + feature_index]
        theta_min = np.min(theta_values)
        theta_max = np.max(theta_values)
        positions = self.data[:, 0]
        unique_positions = np.unique(positions)
        print("🔍 Unique Positions:", unique_positions)

        
         # ✅ Fix the function call (Remove `scenario=` part)
        simulated_clicks = [
    CTRDataset.simulate_clicks(p, r, theta, self.scenario, theta_min, theta_max)
    for r, p, theta in zip(self.data[:, 1], self.data[:, 0], theta_values)
]


        self.labels = torch.tensor(simulated_clicks, dtype=torch.float32).unsqueeze(1).to(self.device)
        self.qids = torch.tensor(self.qids, dtype=torch.int64).to(self.device)  # ✅

    def load_data(self, file_path):
        data = []
        qid_list = []  # <-- Track query IDs
        query_groups = {}

        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                label = int(parts[0])
                query_id = int(parts[1].split(":")[1])
                position = len(query_groups.get(query_id, [])) + 1

                features = np.zeros(136)
                for item in parts[2:]:
                    index, value = item.split(":")
                    features[int(index) - 1] = float(value)

                if query_id not in query_groups:
                    query_groups[query_id] = []
                query_groups[query_id].append([position, label] + list(features))

        # ✅ Add this just before the for-loop over query groups
        if self.ranked_docs:
            print("✅ Using production ranker to reorder documents per query.")

        
        for qid, docs in query_groups.items():
            if self.ranked_docs and qid in self.ranked_docs:
                #print(f"✅ Using production ranker to reorder query {qid}")
                ranked_indices = self.ranked_docs[qid]
                docs = [docs[i] for i in ranked_indices]
            # else: keep original order

            # Sample top 5 + K others
            if len(docs) <= 5:
                sampled_docs = docs
            else:
                top_5 = docs[:5]
                extra_docs = random.sample(docs[5:], min(self.K, len(docs) - 5))
                sampled_docs = top_5 + extra_docs

            # ✅ Reassign positions to avoid 0.0 issues
            for new_pos, doc in enumerate(sampled_docs, start=1):
                doc[0] = float(new_pos)

            data.extend(sampled_docs)
            qid_list.extend([qid] * len(sampled_docs))

        return np.array(data), np.array(qid_list)

    @staticmethod
    def linear_observation_weight(theta, theta_min, theta_max):
        """
        Linearly transforms theta to the range [0.5, 1].
        """
        normalized = (theta - theta_min) / (theta_max - theta_min + 1e-8)  # Prevent div by zero
        return 0.5 + normalized


    def simulate_clicks(position, relevance, theta, scenario, theta_min, theta_max):
        """
        Simulates clicks based on position, relevance, and click simulation scenario.
    
        Parameters:
        relevance (int): Document relevance score (0-4).
        position (int): Document position in search results (1-based index).
        scenario (int): Click simulation scenario (1-5).
        theta (float): Parameter for scenario S5 (document effect).
        title_length (float): Title length feature (only for S5).
        
        Returns:
            int: Simulated click (1 if clicked, 0 otherwise).
        """
    # Step 1: Compute Observation Probability (P(o_d = 1 | pos_d))
        if position <= 0:
            raise ValueError(f"Invalid position: {position}")

        
        if scenario <= 4:  # S1-S4: Observation depends only on position
            position = max(position, 1)  # Ensure position is at least 1
            observation_prob = 1 / position if position <= 5 else 0.1
        else:  # Scenario 5
            omega_theta = CTRDataset.linear_observation_weight(theta, theta_min, theta_max)
            position = max(position, 1)  # Prevent divide-by-zero
            observation_prob = omega_theta / position if position <= 5 else 0.1 * omega_theta

        observed = np.random.rand() < observation_prob 

        # Step 2: Compute Click Probability (P(c_d = 1 | pos_d, r_d, o_d))
        if scenario == 1:
            click_prob = 1 if relevance == 1 and observed else 0

        elif scenario == 2:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else 0

        elif scenario == 3:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0)

        elif scenario == 4:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0.01)

        elif scenario == 5:
            click_prob = 1 if relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if relevance == 1 else 0)

        # Generate Click
        click = np.random.rand() < click_prob
        return int(click)

    

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Returns (features, bias, position, label)"""
        return self.features[idx], self.bias_features[idx], self.position[idx], self.labels[idx],self.qids[idx]  # ✅ Return qid

In [6]:
def train_production_ranker(dataset, train_ratio=0.1, max_attempts=50):
    """
    Train a simple production ranker using a small subset of training data.
    
    Ensures at least one `1` and one `0` are always included.
    """
    dataloader = DataLoader(dataset, batch_size=len(dataset))  # Load all data at once
    features, _, _, relevance, _ = next(iter(dataloader))


    features = features.cpu().numpy()
    relevance = relevance.cpu().numpy().flatten()

    # ✅ Ensure dataset has both `1` and `0` labels
    ones = [i for i, r in enumerate(relevance) if r == 1]
    zeros = [i for i, r in enumerate(relevance) if r == 0]

    if len(ones) == 0 or len(zeros) == 0:
        print("❌ No positive or negative samples found in dataset!")
        return None, None

    for attempt in range(max_attempts):
        sample_size = max(100, int(len(features) * train_ratio))  # Ensure at least 100 samples
        sample_size = min(sample_size, len(features))  # Avoid exceeding dataset size

        # ✅ Always include at least one `1` and one `0`
        num_ones = min(len(ones), sample_size // 2)
        num_zeros = min(len(zeros), sample_size - num_ones)

        sampled_indices = random.sample(ones, num_ones) + random.sample(zeros, num_zeros)
        train_features = features[sampled_indices]
        train_labels = np.array([1 if r >= 1 else 0 for r in relevance[sampled_indices]])

        label_counts = Counter(train_labels)
        if len(label_counts) > 1:
            break  # ✅ Valid dataset found
        print(f"⚠️ Re-sampling attempt {attempt + 1}: Only one class found, increasing sample size...")

    else:
        print("❌ Failed to find a balanced sample after multiple attempts. Returning None.")
        return None, None

    # Scale features
    scaler = MinMaxScaler()
    train_features = scaler.fit_transform(train_features)

    # Train logistic regression model
    model = LogisticRegression(random_state=42, class_weight='balanced', solver='saga', max_iter=500)
    model.fit(train_features, train_labels)

    print(f"✅ Production Ranker Trained on {len(train_features)} Samples")
    return model, scaler


In [7]:
from collections import defaultdict
import numpy as np

def rank_documents(model, scaler, dataset):
    """
    Rank documents per query using the trained production ranker.
    Returns a dictionary: {qid: [doc_indices_sorted_by_score]}
    """
    dataloader = DataLoader(dataset, batch_size=len(dataset))
    features, _, _, relevance, qid = next(iter(dataloader))

    # Convert tensors to numpy
    features = features.cpu().numpy()
    qid = qid.cpu().numpy()

    # Scale features
    features_scaled = scaler.transform(features)

    # Predict relevance scores
    scores = model.predict_proba(features_scaled)[:, 1]

    # Group document indices by qid
    qid_to_indices = defaultdict(list)
    for idx, q in enumerate(qid):
        qid_to_indices[q].append(idx)

    # Rank within each query
    ranked_docs = {}
    for q, doc_indices in qid_to_indices.items():
        # Pair each document's local index (0 to len-1) with its score
        local_doc_scores = [(i, scores[doc_indices[i]]) for i in range(len(doc_indices))]
        local_doc_scores.sort(key=lambda x: -x[1])  # Sort by score descending
        ranked_docs[q] = [i for i, _ in local_doc_scores]  # local indices

    """ for q, doc_indices in qid_to_indices.items():
        doc_scores = [(idx, scores[idx]) for idx in doc_indices]
        doc_scores.sort(key=lambda x: -x[1])  # Sort by score descending
        ranked_docs[q] = [idx for idx, _ in doc_scores]"""

    return ranked_docs


In [ ]:
# ✅ Step 1: Load the Full Training Dataset (Real Labels)
train_file = "C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\train.txt"
train_dataset_full = CTRDataset(train_file, scenario=1, device=device)

# ✅ Check label distribution before training
dataloader = DataLoader(train_dataset_full, batch_size=len(train_dataset_full))  # Load all data at once
features, bias, position, relevance, qid = next(iter(dataloader))

full_labels = relevance.flatten().cpu().numpy()

label_counts = Counter(full_labels)

print(f"🔍 Label distribution in full training set: {label_counts}")

# ✅ Step 2: Train the Production Ranker on a 1% Subset
print("🔹 Training Production Ranker...")
production_ranker, scaler = train_production_ranker(train_dataset_full, train_ratio=0.05)

# ✅ Step 3: Check if Ranker & Scaler Are Valid
if production_ranker is None or scaler is None:
    print("❌ Production Ranker Training Failed! Using Default Labels Instead.")
    use_ranker = False  # Disable ranker-based ranking
else:
    print("✅ Production Ranker Successfully Trained.")
    use_ranker = True  # Enable ranker-based ranking

# ✅ Step 4: Generate Initial Document Rankings (`R̂_q`) Only If Ranker Exists
if use_ranker:
    print("🔹 Ranking Documents Using Production Ranker...")
    ranked_docs = rank_documents(production_ranker, scaler, train_dataset_full)
else:
    print("none")
    ranked_docs = None  # Skip ranking if ranker failed


# ✅ Step 5: Train the Main Model Using Simulated Clicks
print("🔹 Simulating Clicks and Training Model...")

# Load Training Data with Click Simulation (Using Ranked Documents)
train_dataset = CTRDataset(train_file, scenario=1, device=device, ranked_docs=ranked_docs)


train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True)


# Train the Model with Simulated Click Labels
#train_model(model, train_dataloader, optimizer, epochs=70, device=device)

# ✅ Step 6: Load Test Data (Real Labels: Relevance ≥ 2 → 1)
test_file = "C:\\Users\\Amala K J\\Downloads\\MSLR-WEB30K\\Fold1\\test.txt"
test_dataset = CTRDataset(test_file, scenario=1, device=device)  # Test data should NOT be simulated!
test_dataloader = DataLoader(test_dataset, batch_size=256, shuffle=False)


# Instead of your simple training loop, use this single line:
#analyzer, results, best_config, final_model = run_enhanced_comprehensive_analysis( train_dataloader, test_dataloader, device=device)


# Replace this line:
# analyzer, results, best_config, final_model = run_enhanced_comprehensive_analysis(
#     train_dataloader, test_dataloader, device=device
# )

# With this much faster version:
#analyzer, results, best_config, final_model = run_quick_comprehensive_analysis(
#    train_dataloader, test_dataloader, device=device
#)

analyzer, results, best_config = run_minimal_analysis(
    train_dataloader, test_dataloader, device=device
)
# ✅ Step 7: Evaluate Model on Real Test Data
evaluate_model(model, test_dataloader, device=device)
# After training
#evaluate_ndcg_k_range(model, test_dataloader, device=device)


